In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Paste your directory path in the 'dir' variable**

In [2]:
import os
dir = '/content/drive/MyDrive/Colab Notebooks/Fake Review Detection'
os.chdir(dir)

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
import numpy as np
import joblib

**Paste the directory path of preprocessed data in the variable 'csv_path'**

In [4]:
# Step 1: Load the dataset
csv_path = "./Preprocessed Data/fake reviews dataset_final_text_preprocessed_v3.csv"  # Replace with the correct file path
df = pd.read_csv(csv_path)

In [5]:
# Step 2: Handle missing values in the 'text' column
df = df.dropna(subset=['text'])  # Drop rows with NaN in 'text'
df['text'] = df['text'].fillna('')  # Alternatively, replace NaN with an empty string

In [6]:
df.head()

,Unnamed: 0,label,text
0,0,1,good canned asparagus used get canned asparagu...
1,1,1,didnt buy particular one bought bigger one sal...
2,2,0,supposedly great chew toy problem kind hard pu...
3,3,1,wanted try new coffee company found brooklyn c...
4,4,1,dont like ginger youre going like gold kilis g...


In [7]:
# Check for required columns
if 'label' not in df.columns or 'text' not in df.columns:
    raise ValueError("The dataset must contain 'label' and 'text' columns.")

# Step 2: Split the dataset into train, validation, and test sets
train_text, temp_text, train_labels, temp_labels = train_test_split(
    df['text'], df['label'], test_size=0.30, random_state=42
)

val_text, test_text, val_labels, test_labels = train_test_split(
    temp_text, temp_labels, test_size=0.33, random_state=42
)

In [ ]:
# Step 4: Define pipeline and train the model
pipeline = Pipeline([
    ('bow', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('classifier', MultinomialNB())
])

param_grid = {'classifier__alpha': [0.1, 0.5, 1, 5, 10]}
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='accuracy', verbose=2)
grid_search.fit(train_text, train_labels)


model_filename = "trained_model_pipeline.pkl"
joblib.dump(grid_search.best_estimator_, model_filename)
print(f"Model saved as {model_filename}")

Fitting 3 folds for each of 5 candidates, totalling 15 fits
[CV] END ..............................classifier__alpha=0.1; total time=  19.3s
[CV] END ..............................classifier__alpha=0.1; total time=  19.2s
[CV] END ..............................classifier__alpha=0.1; total time=  19.3s
[CV] END ..............................classifier__alpha=0.5; total time=  19.2s
[CV] END ..............................classifier__alpha=0.5; total time=  19.1s
[CV] END ..............................classifier__alpha=0.5; total time=  19.0s
[CV] END ................................classifier__alpha=1; total time=  19.2s
[CV] END ................................classifier__alpha=1; total time=  19.0s
[CV] END ................................classifier__alpha=1; total time=  19.0s
[CV] END ................................classifier__alpha=5; total time=  19.1s
[CV] END ................................classifier__alpha=5; total time=  18.8s
[CV] END ................................classifi

In [ ]:
val_predictions = grid_search.best_estimator_.predict(val_text)
val_accuracy = accuracy_score(val_labels, val_predictions)
print("\nValidation Accuracy:", val_accuracy)


Validation Accuracy: 0.7762784130859101


In [ ]:
# Step 9: Evaluate the model on the test set
test_predictions = grid_search.best_estimator_.predict(test_text)
test_accuracy = accuracy_score(test_labels, test_predictions)
print("\nTest Accuracy:", test_accuracy)
print("\nTest Classification Report:\n", classification_report(test_labels, test_predictions))


Test Accuracy: 0.7772460970958053

Test Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.34      0.50     29227
           1       0.75      1.00      0.86     58336

    accuracy                           0.78     87563
   macro avg       0.87      0.67      0.68     87563
weighted avg       0.83      0.78      0.74     87563



In [ ]:
# Load the saved model
model_filename = "trained_model_pipeline.pkl"
loaded_model = joblib.load(model_filename)
print(f"Model loaded from {model_filename}")

# Generate predictions for the training set
train_predictions = loaded_model.predict(train_text)

# Calculate training accuracy
train_accuracy = accuracy_score(train_labels, train_predictions)
print(f"Training Accuracy: {train_accuracy:.4f}")

Model loaded from trained_model_pipeline.pkl
Training Accuracy: 0.7789
